In [ ]:
# Parameters (will be overridden by URL query params when running with Voila)
import os
import json
from datetime import datetime

# Default fallback Transaction ID
transactionId = os.getenv("CONDITIONS_TRANSACTION_ID", "TMICFBPK2801321903297120")
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3090")
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
#headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET} if JUPYTER_SHARED_SECRET else {}
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}
tenantId = os.getenv("CONDITIONS_TENANT_ID", "DEFAULT")

In [ ]:
import requests
from IPython.display import display, HTML

def fetch_json(endpoint: str, params: dict):
    url = f"{backendUrl}{endpoint}"
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()

# Fetch transaction context
context_data = {}
try:
    base_params = {
        'tenantId': tenantId,
    }
    context_data = fetch_json(
        f"/api/v1/jupyter/proxy/conditions/by-transaction/{transactionId}",
        {'tenantId': tenantId},
    )
except Exception as e:
    display(HTML(f"<div style='color: red; padding: 20px;'>Error: {str(e)}</div>"))

In [ ]:
if context_data:
    tx = context_data.get('transaction', {})
    debtor = context_data.get('debtor', {})
    creditor = context_data.get('creditor', {})
    metadata = context_data.get('metadata', {})
    
    # Fetch conditions for debtor's primary account
    debtor_account_id = debtor.get('primaryAccountId') or (debtor.get('accounts', [{}])[0].get('accountId') if debtor.get('accounts') else '')
    creditor_account_id = creditor.get('primaryAccountId') or (creditor.get('accounts', [{}])[0].get('accountId') if creditor.get('accounts') else '')
    
    debtor_conditions = []
    creditor_conditions = []
    
    try:
        if debtor_account_id:
            debtor_details = fetch_json("/api/v1/jupyter/proxy/conditions/details", 
                {'accountId': debtor_account_id, 'tenantId': tenantId, 'showInactive': 'true'})
            debtor_conditions = debtor_details.get('conditions', [])
    except:
        pass
    
    try:
        if creditor_account_id:
            creditor_details = fetch_json("/api/v1/jupyter/proxy/conditions/details",
                {'accountId': creditor_account_id, 'tenantId': tenantId, 'showInactive': 'true'})
            creditor_conditions = creditor_details.get('conditions', [])
    except:
        pass

In [ ]:
if context_data:
    tx_display_id = tx.get('displayId', f"TXN-{tx.get('transactionId', transactionId)}")
    viewing_as_of = metadata.get('asOfDate', datetime.now().strftime('%d/%m/%Y %I:%M:%S %p'))
    if viewing_as_of and 'T' in str(viewing_as_of):
        try:
            dt = datetime.fromisoformat(viewing_as_of.replace('Z', '+00:00'))
            viewing_as_of = dt.strftime('%d/%m/%Y %I:%M:%S %p').lower()
        except:
            pass
    
    # Build data for JS
    js_data = {
        'transaction': tx,
        'debtor': {
            'entityName': debtor.get('entityName', 'Unknown'),
            'entityId': debtor.get('entityId', '-'),
            'accounts': debtor.get('accounts', []),
            'conditions': debtor_conditions
        },
        'creditor': {
            'entityName': creditor.get('entityName', 'Unknown'),
            'entityId': creditor.get('entityId', '-'),
            'accounts': creditor.get('accounts', []),
            'conditions': creditor_conditions
        }
    }
    
    html_content = f"""
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 0; padding: 0; background: #f8fafc; }}
        * {{ box-sizing: border-box; }}
        
        /* Header */
        .conditions-header {{ display: flex; justify-content: space-between; align-items: center; padding: 16px 24px; background: white; border-bottom: 1px solid #e2e8f0; }}
        .header-left {{ display: flex; align-items: center; gap: 8px; }}
        .header-icon {{ color: #6366f1; }}
        .header-title {{ font-size: 16px; font-weight: 600; color: #0f172a; }}
        .header-subtitle {{ font-size: 12px; color: #64748b; margin-top: 2px; }}
        .viewing-as-of {{ display: flex; align-items: center; gap: 10px; padding: 8px 14px; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; }}
        .viewing-label {{ font-size: 10px; color: #6366f1; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px; }}
        .viewing-date {{ font-size: 13px; color: #0f172a; font-weight: 500; }}
        
        /* Main Layout */
        .main-content {{ display: flex; gap: 0; min-height: 700px; }}
        
        /* Sidebar */
        .sidebar {{ width: 280px; background: white; border-right: 1px solid #e2e8f0; padding: 20px; flex-shrink: 0; }}
        .sidebar-section {{ margin-bottom: 24px; }}
        .sidebar-label {{ font-size: 10px; font-weight: 600; color: #94a3b8; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 12px; }}
        
        /* Toggle Buttons */
        .toggle-group {{ display: flex; background: #f1f5f9; border-radius: 8px; padding: 4px; }}
        .toggle-btn {{ flex: 1; padding: 10px 16px; border: none; background: transparent; font-size: 13px; font-weight: 500; color: #64748b; cursor: pointer; border-radius: 6px; display: flex; align-items: center; justify-content: center; gap: 6px; transition: all 0.15s; }}
        .toggle-btn.active {{ background: white; color: #0f172a; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        .toggle-btn:hover:not(.active) {{ color: #0f172a; }}
        
        /* Level Selection */
        .level-option {{ display: flex; align-items: center; gap: 12px; padding: 12px 14px; border: 1px solid #e2e8f0; border-radius: 10px; margin-bottom: 8px; cursor: pointer; transition: all 0.15s; }}
        .level-option:hover {{ border-color: #cbd5e1; }}
        .level-option.active {{ border-color: #6366f1; background: #f8fafc; }}
        .level-option.active .level-dot {{ background: #6366f1; }}
        .level-icon {{ width: 32px; height: 32px; background: #f1f5f9; border-radius: 8px; display: flex; align-items: center; justify-content: center; color: #64748b; }}
        .level-info {{ flex: 1; }}
        .level-title {{ font-size: 13px; font-weight: 600; color: #0f172a; }}
        .level-subtitle {{ font-size: 11px; color: #94a3b8; }}
        .level-dot {{ width: 10px; height: 10px; border-radius: 50%; border: 2px solid #e2e8f0; }}
        
        /* Account Selector */
        .account-selector {{ display: none; margin-top: 16px; }}
        .account-selector.visible {{ display: block; }}
        .account-option {{ display: flex; align-items: center; gap: 12px; padding: 12px 14px; border: 1px solid #e2e8f0; border-radius: 10px; margin-bottom: 8px; cursor: pointer; transition: all 0.15s; }}
        .account-option:hover {{ border-color: #cbd5e1; background: #fafafa; }}
        .account-option.active {{ border-color: #6366f1; background: #f8fafc; }}
        .account-icon {{ width: 32px; height: 32px; background: #f1f5f9; border-radius: 8px; display: flex; align-items: center; justify-content: center; color: #64748b; }}
        .account-info {{ flex: 1; }}
        .account-type {{ font-size: 13px; font-weight: 600; color: #0f172a; display: flex; align-items: center; gap: 8px; }}
        .account-number {{ font-size: 11px; color: #94a3b8; font-family: monospace; }}
        .this-account-badge {{ background: #dbeafe; color: #2563eb; font-size: 9px; font-weight: 600; padding: 2px 6px; border-radius: 4px; text-transform: uppercase; }}
        .account-condition-count {{ font-size: 12px; font-weight: 600; color: #64748b; }}
        
        /* Subject Details Card */
        .details-card {{ background: #f8fafc; border-radius: 10px; padding: 16px; }}
        .detail-row {{ margin-bottom: 12px; }}
        .detail-row:last-child {{ margin-bottom: 0; }}
        .detail-label {{ font-size: 10px; color: #94a3b8; text-transform: uppercase; margin-bottom: 4px; }}
        .detail-value {{ font-size: 14px; font-weight: 600; color: #0f172a; }}
        .detail-badge {{ display: inline-block; background: #f1f5f9; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-family: monospace; color: #475569; }}
        
        /* Active Blocks */
        .blocks-indicator {{ display: flex; justify-content: space-between; align-items: center; padding-top: 16px; border-top: 1px solid #e2e8f0; margin-top: 16px; }}
        .blocks-label {{ font-size: 13px; color: #64748b; }}
        .blocks-count {{ font-size: 18px; font-weight: 700; color: #0f172a; }}
        .blocks-count.has-blocks {{ color: #ef4444; }}
        
        /* Content Area */
        .content-area {{ flex: 1; padding: 24px; background: #f8fafc; min-height: 700px; }}
        
        /* Conditions Header */
        .conditions-list-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px; }}
        .conditions-title {{ font-size: 15px; font-weight: 600; color: #0f172a; }}
        .conditions-count {{ color: #94a3b8; font-weight: 400; }}
        .show-inactive {{ display: flex; align-items: center; gap: 8px; }}
        .show-inactive-label {{ font-size: 13px; color: #64748b; }}
        .toggle-switch {{ position: relative; width: 44px; height: 24px; background: #e2e8f0; border-radius: 12px; cursor: pointer; transition: background 0.2s; }}
        .toggle-switch.active {{ background: #6366f1; }}
        .toggle-switch::after {{ content: ''; position: absolute; width: 20px; height: 20px; background: white; border-radius: 50%; top: 2px; left: 2px; transition: transform 0.2s; box-shadow: 0 1px 3px rgba(0,0,0,0.2); }}
        .toggle-switch.active::after {{ transform: translateX(20px); }}
        
        /* Condition Card */
        .condition-card {{ background: white; border-radius: 12px; padding: 20px; margin-bottom: 16px; border-left: 4px solid; position: relative; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }}
        .condition-card.block {{ border-left-color: #ef4444; }}
        .condition-card.override {{ border-left-color: #22c55e; }}
        .condition-card.expired {{ border-left-color: #94a3b8; opacity: 0.7; }}
        .condition-card-header {{ display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 8px; }}
        .condition-icon {{ width: 28px; height: 28px; border-radius: 50%; display: flex; align-items: center; justify-content: center; margin-right: 12px; flex-shrink: 0; }}
        .condition-icon.block {{ background: #fee2e2; color: #ef4444; }}
        .condition-icon.override {{ background: #dcfce7; color: #22c55e; }}
        .condition-title-row {{ display: flex; align-items: center; }}
        .condition-title {{ font-size: 15px; font-weight: 600; color: #0f172a; }}
        .condition-desc {{ font-size: 13px; color: #64748b; margin-top: 4px; margin-left: 40px; }}
        .condition-source {{ text-align: right; }}
        .source-label {{ font-size: 10px; color: #94a3b8; text-transform: uppercase; }}
        .source-value {{ font-size: 12px; color: #475569; font-weight: 500; }}
        .condition-footer {{ display: flex; justify-content: space-between; align-items: center; margin-top: 16px; padding-top: 12px; border-top: 1px solid #f1f5f9; flex-wrap: wrap; gap: 12px; }}
        .condition-dates {{ display: flex; align-items: center; gap: 6px; font-size: 12px; color: #64748b; }}
        .condition-note {{ background: #f8fafc; padding: 8px 12px; border-radius: 6px; font-size: 12px; color: #475569; }}
        .note-label {{ font-size: 10px; color: #94a3b8; font-weight: 600; margin-right: 6px; }}
        
        /* Empty State */
        .empty-state {{ text-align: center; padding: 60px 20px; color: #94a3b8; }}
        .empty-icon {{ font-size: 48px; margin-bottom: 16px; }}
        .empty-text {{ font-size: 14px; }}
    </style>
    
    <div class=\"conditions-container\">
        <div class=\"conditions-header\">
            <div class=\"header-left\">
                <svg class=\"header-icon\" width=\"20\" height=\"20\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                    <circle cx=\"12\" cy=\"12\" r=\"10\"></circle>
                    <polyline points=\"12,6 12,12 16,14\"></polyline>
                </svg>
                <div>
                    <div class=\"header-title\">Condition Timeline</div>
                    <div class=\"header-subtitle\">Transaction: {tx_display_id}</div>
                </div>
            </div>
            <div class=\"viewing-as-of\">
                <svg width=\"16\" height=\"16\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#6366f1\" stroke-width=\"2\">
                    <circle cx=\"12\" cy=\"12\" r=\"10\"></circle>
                    <polyline points=\"12,6 12,12 16,14\"></polyline>
                </svg>
                <div>
                    <div class=\"viewing-label\">Viewing Conditions As Of</div>
                    <div class=\"viewing-date\">{viewing_as_of}</div>
                </div>
                <svg width=\"16\" height=\"16\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#94a3b8\" stroke-width=\"2\">
                    <rect x=\"3\" y=\"4\" width=\"18\" height=\"18\" rx=\"2\" ry=\"2\"></rect>
                    <line x1=\"16\" y1=\"2\" x2=\"16\" y2=\"6\"></line>
                    <line x1=\"8\" y1=\"2\" x2=\"8\" y2=\"6\"></line>
                    <line x1=\"3\" y1=\"10\" x2=\"21\" y2=\"10\"></line>
                </svg>
            </div>
        </div>
        
        <div class=\"main-content\">
            <div class=\"sidebar\">
                <div class=\"sidebar-section\">
                    <div class=\"sidebar-label\">Subject Selection</div>
                    <div class=\"toggle-group\">
                        <button class=\"toggle-btn active\" id=\"btn-debtor\" onclick=\"selectSide('debtor')\">
                            <svg width=\"14\" height=\"14\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                                <path d=\"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2\"></path>
                                <circle cx=\"12\" cy=\"7\" r=\"4\"></circle>
                            </svg>
                            Debtor
                        </button>
                        <button class=\"toggle-btn\" id=\"btn-creditor\" onclick=\"selectSide('creditor')\">
                            <svg width=\"14\" height=\"14\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                                <rect x=\"2\" y=\"5\" width=\"20\" height=\"14\" rx=\"2\"></rect>
                                <line x1=\"2\" y1=\"10\" x2=\"22\" y2=\"10\"></line>
                            </svg>
                            Creditor
                        </button>
                    </div>
                </div>
                
                <div class=\"sidebar-section\">
                    <div class=\"level-option active\" id=\"level-entity\" onclick=\"selectLevel('entity')\">
                        <div class=\"level-icon\">
                            <svg width=\"16\" height=\"16\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                                <path d=\"M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2\"></path>
                                <circle cx=\"9\" cy=\"7\" r=\"4\"></circle>
                                <path d=\"M23 21v-2a4 4 0 0 0-3-3.87\"></path>
                                <path d=\"M16 3.13a4 4 0 0 1 0 7.75\"></path>
                            </svg>
                        </div>
                        <div class=\"level-info\">
                            <div class=\"level-title\">Entity Level</div>
                            <div class=\"level-subtitle\">Legal Entity Conditions</div>
                        </div>
                        <div class=\"level-dot\"></div>
                    </div>
                    <div class=\"level-option\" id=\"level-account\" onclick=\"selectLevel('account')\">
                        <div class=\"level-icon\">
                            <svg width=\"16\" height=\"16\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                                <rect x=\"1\" y=\"4\" width=\"22\" height=\"16\" rx=\"2\" ry=\"2\"></rect>
                                <line x1=\"1\" y1=\"10\" x2=\"23\" y2=\"10\"></line>
                            </svg>
                        </div>
                        <div class=\"level-info\">
                            <div class=\"level-title\">Account Level</div>
                            <div class=\"level-subtitle\" id=\"account-count\">Loading...</div>
                        </div>
                        <div class=\"level-dot\"></div>
                    </div>
                    
                    <div class=\"account-selector\" id=\"account-selector\">
                        <div class=\"sidebar-label\">Select Account</div>
                        <div id=\"account-list\"></div>
                    </div>
                </div>
                
                <div class=\"sidebar-section\">
                    <div class=\"sidebar-label\">Selected Subject Details</div>
                    <div class=\"details-card\">
                        <div class=\"detail-row\">
                            <div class=\"detail-label\">Entity Name</div>
                            <div class=\"detail-value\" id=\"entity-name\">Loading...</div>
                        </div>
                        <div class=\"detail-row\">
                            <div class=\"detail-label\">Entity ID</div>
                            <div class=\"detail-badge\" id=\"entity-id\">-</div>
                        </div>
                        <div class=\"detail-row\" id=\"account-type-row\" style=\"display: none;\">
                            <div class=\"detail-label\">Account Type</div>
                            <div class=\"detail-value\" id=\"account-type\">-</div>
                        </div>
                        <div class=\"detail-row\" id=\"account-id-row\" style=\"display: none;\">
                            <div class=\"detail-label\">Account ID</div>
                            <div class=\"detail-badge\" id=\"account-id-display\">-</div>
                        </div>
                        <div class=\"detail-row\" id=\"account-number-row\" style=\"display: none;\">
                            <div class=\"detail-label\">Account Number</div>
                            <div class=\"detail-badge\" id=\"account-number-display\">-</div>
                        </div>
                        <div class=\"blocks-indicator\">
                            <div class=\"blocks-label\">Active Blocks</div>
                            <div class=\"blocks-count\" id=\"blocks-count\">0</div>
                        </div>
                    </div>
                </div>
            </div>
            
            <div class=\"content-area\">
                <div class=\"conditions-list-header\">
                    <div class=\"conditions-title\">Conditions <span class=\"conditions-count\" id=\"conditions-count\">(0 active)</span></div>
                    <div class=\"show-inactive\">
                        <span class=\"show-inactive-label\">Show Inactive</span>
                        <div class=\"toggle-switch\" id=\"toggle-inactive\" onclick=\"toggleInactive()\"></div>
                    </div>
                </div>
                <div id=\"conditions-list\"></div>
            </div>
        </div>
    </div>
    
    <script>
    (function() {{
        const data = {json.dumps(js_data)};
        let currentSide = 'debtor';
        let currentLevel = 'entity';
        let selectedAccountIdx = 0;
        let showInactive = false;
        
        function getParty() {{
            return currentSide === 'debtor' ? data.debtor : data.creditor;
        }}
        
        function formatAccountType(acc) {{
            if (!acc) return 'Account';
            if (acc.isTransactionAccount) return 'TXN ACCOUNT';
            return acc.accountType || acc.type || 'Account';
        }}
        
        function formatAccountNumber(acc) {{
            if (!acc) return '-';
            return acc.accountNumber || '-';
        }}
        
        function updateUI() {{
            const party = getParty();
            const accounts = party.accounts || [];
            
            // Update subject details
            document.getElementById('entity-name').textContent = party.entityName || 'Unknown';
            document.getElementById('entity-id').textContent = party.entityId || '-';
            
            // Update account count
            document.getElementById('account-count').textContent = accounts.length + ' accounts available';
            
            // Show/hide account selector
            const accountSelector = document.getElementById('account-selector');
            if (currentLevel === 'account') {{
                accountSelector.classList.add('visible');
                renderAccountList(accounts);
                
                // Show account details
                document.getElementById('account-type-row').style.display = 'block';
                document.getElementById('account-id-row').style.display = 'block';
                document.getElementById('account-number-row').style.display = 'block';
                
                if (accounts.length > 0 && selectedAccountIdx < accounts.length) {{
                    const acc = accounts[selectedAccountIdx];
                    document.getElementById('account-type').textContent = formatAccountType(acc);
                    document.getElementById('account-id-display').textContent = acc.accountId || '-';
                    document.getElementById('account-number-display').textContent = formatAccountNumber(acc);
                }}
            }} else {{
                accountSelector.classList.remove('visible');
                document.getElementById('account-type-row').style.display = 'none';
                document.getElementById('account-id-row').style.display = 'none';
                document.getElementById('account-number-row').style.display = 'none';
            }}
            
            // Get conditions
            const conditions = party.conditions || [];
            const activeConditions = conditions.filter(c => c.status === 'active' || (c.isActive && !c.isExpired));
            const displayConditions = showInactive ? conditions : activeConditions;
            
            // Count blocks
            const blockCount = activeConditions.filter(c => (c.type || '').toLowerCase().includes('block')).length;
            const blocksEl = document.getElementById('blocks-count');
            blocksEl.textContent = blockCount;
            blocksEl.classList.toggle('has-blocks', blockCount > 0);
            
            // Update count
            document.getElementById('conditions-count').textContent = '(' + activeConditions.length + ' active)';
            
            // Render conditions
            renderConditions(displayConditions);
        }}
        
        function renderAccountList(accounts) {{
            const container = document.getElementById('account-list');
            if (!accounts || accounts.length === 0) {{
                container.innerHTML = '<div style=\"color: #94a3b8; font-size: 12px; padding: 10px;\">No accounts available</div>';
                return;
            }}
            
            let html = '';
            accounts.forEach((acc, idx) => {{
                const isActive = idx === selectedAccountIdx;
                const accType = formatAccountType(acc);
                const accNum = formatAccountNumber(acc);
                const condCount = acc.conditionCount || 0;
                
                html += `
                <div class=\"account-option ${{isActive ? 'active' : ''}}\" onclick=\"selectAccount(${{idx}})\">
                    <div class=\"account-icon\">
                        <svg width=\"16\" height=\"16\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                            <rect x=\"1\" y=\"4\" width=\"22\" height=\"16\" rx=\"2\" ry=\"2\"></rect>
                            <line x1=\"1\" y1=\"10\" x2=\"23\" y2=\"10\"></line>
                        </svg>
                    </div>
                    <div class=\"account-info\">
                        <div class=\"account-type\">
                            ${{accType}}
                        </div>
                        <div class=\"account-number\">${{accNum}}</div>
                    </div>
                    <div class=\"account-condition-count\">${{condCount}}</div>
                </div>`;
            }});
            
            container.innerHTML = html;
        }}
        
        function renderConditions(conditions) {{
            const container = document.getElementById('conditions-list');
            
            if (!conditions || conditions.length === 0) {{
                container.innerHTML = '<div class=\"empty-state\"><div class=\"empty-icon\">✓</div><div class=\"empty-text\">No conditions found for this subject</div></div>';
                return;
            }}
            
            let html = '';
            conditions.forEach(c => {{
                const isBlock = (c.type || '').toLowerCase().includes('block');
                const isExpired = c.status === 'expired' || c.isExpired;
                const cardClass = isExpired ? 'expired' : (isBlock ? 'block' : 'override');
                const iconClass = isBlock ? 'block' : 'override';
                const iconSvg = isBlock 
                    ? '<svg width=\"14\" height=\"14\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\"><circle cx=\"12\" cy=\"12\" r=\"10\"></circle><line x1=\"4.93\" y1=\"4.93\" x2=\"19.07\" y2=\"19.07\"></line></svg>'
                    : '<svg width=\"14\" height=\"14\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\"><polyline points=\"20,6 9,17 4,12\"></polyline></svg>';
                
                const inception = c.inceptionTimestamp || c.inceptionDate || '-';
                const expiry = c.expiryTimestamp || c.expiryDate || 'Indefinite';
                
                let inceptionFormatted = inception;
                let expiryFormatted = expiry;
                try {{
                    if (inception && inception !== '-') {{
                        const d = new Date(inception);
                        inceptionFormatted = d.toLocaleDateString('en-US', {{ month: 'short', day: 'numeric', year: 'numeric' }}) + ', ' + d.toLocaleTimeString('en-US', {{ hour: 'numeric', minute: '2-digit', hour12: true }});
                    }}
                    if (expiry && expiry !== 'Indefinite' && expiry !== '-') {{
                        const d = new Date(expiry);
                        expiryFormatted = d.toLocaleDateString('en-US', {{ month: 'short', day: 'numeric', year: 'numeric' }}) + ', ' + d.toLocaleTimeString('en-US', {{ hour: 'numeric', minute: '2-digit', hour12: true }});
                    }}
                }} catch(e) {{}}
                
                html += `
                <div class=\"condition-card ${{cardClass}}\">
                    <div class=\"condition-card-header\">
                        <div>
                            <div class=\"condition-title-row\">
                                <div class=\"condition-icon ${{iconClass}}\">${{iconSvg}}</div>
                                <div class=\"condition-title\">${{c.type || 'Condition'}}</div>
                            </div>
                            <div class=\"condition-desc\">${{c.description || c.reason || '-'}}</div>
                        </div>
                        <div class=\"condition-source\">
                            <div class=\"source-label\">Source</div>
                            <div class=\"source-value\">${{c.createdBy || c.source || 'Manual Override'}}</div>
                        </div>
                    </div>
                    <div class=\"condition-footer\">
                        <div class=\"condition-dates\">
                            <svg width=\"14\" height=\"14\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">
                                <rect x=\"3\" y=\"4\" width=\"18\" height=\"18\" rx=\"2\" ry=\"2\"></rect>
                                <line x1=\"16\" y1=\"2\" x2=\"16\" y2=\"6\"></line>
                                <line x1=\"8\" y1=\"2\" x2=\"8\" y2=\"6\"></line>
                                <line x1=\"3\" y1=\"10\" x2=\"21\" y2=\"10\"></line>
                            </svg>
                            Effective: ${{inceptionFormatted}} → ${{expiryFormatted}}
                        </div>
                        ${{c.note || c.reason ? `<div class=\"condition-note\"><span class=\"note-label\">NOTE:</span>${{c.note || c.reason}}</div>` : ''}}
                    </div>
                </div>`;
            }});
            
            container.innerHTML = html;
        }}
        
        window.selectSide = function(side) {{
            currentSide = side;
            selectedAccountIdx = 0;
            document.getElementById('btn-debtor').classList.toggle('active', side === 'debtor');
            document.getElementById('btn-creditor').classList.toggle('active', side === 'creditor');
            updateUI();
        }};
        
        window.selectLevel = function(level) {{
            currentLevel = level;
            selectedAccountIdx = 0;
            document.getElementById('level-entity').classList.toggle('active', level === 'entity');
            document.getElementById('level-account').classList.toggle('active', level === 'account');
            updateUI();
        }};
        
        window.selectAccount = function(idx) {{
            selectedAccountIdx = idx;
            updateUI();
        }};
        
        window.toggleInactive = function() {{
            showInactive = !showInactive;
            document.getElementById('toggle-inactive').classList.toggle('active', showInactive);
            updateUI();
        }};
        
        // Initial render
        updateUI();
    }})();
    </script>
    """
    
    display(HTML(html_content))
else:
    display(HTML("<div style='padding: 40px; text-align: center; color: #94a3b8;'>No data available</div>"))


In [ ]:
from IPython.display import HTML, display

display(
    HTML(
        """
<style>
  /* Layout: make left sidebar a bit wider */
  .sidebar { width: 340px !important; }

  /* Prevent long IDs from being clipped */
  .detail-badge {
    display: block !important;
    max-width: 100% !important;
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: break-word !important;
  }

  /* Same for account numbers / monospace labels */
  .account-number {
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: break-word !important;
  }

  /* Hide the "No conditions found" empty state */
  .empty-state { display: none !important; }
</style>
        """
    )
)
